# Capstone Project: Video Search Dashboard

This notebook implements the full solution for the NLP capstone lab using AWS services.
Each section corresponds exactly to the lab instructions.


## AWS Capstone Lab Context (from the lab instructions)

**Business scenario:** You work for a training organization that developed an introductory machine learning course with 40+ videos.
You need to create an application that helps students quickly locate and view video content by searching for topics and key phrases.
All source videos are stored in an Amazon S3 bucket.

**Lab steps (mapped to sections below):**
1. **Viewing the video files** (S3 listing)
2. **Transcribing the videos** (Amazon Transcribe)
3. **Normalizing the text** (cleanup for NLP)
4. **Extracting key phrases and topics** (Amazon Comprehend)
5. **Creating the dashboard** (search experience for students)

> **Note:** In the lab console, choose **Submit** to record your progress.


## 1.3 Useful Information

In [ ]:

# Replace ONLY if your lab provides different values
bucket = "c183398a4725384l13434991t1w746048835797-labbucket-hjpbs2li2szl"
job_data_access_role = "arn:aws:iam::746048835797:role/service-role/c183398a4725384l13434991t1-ComprehendDataAccessRole-ttSR5Iviz1vZ"


## 1.4 Viewing the video files

In [ ]:
# AWS Step 1 (Viewing videos):
# We list the shared S3 bucket to confirm which MP4 files are available for processing.

# List available training videos in the shared S3 bucket
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/


## 1.5 Transcribing the videos

In [ ]:
# AWS Step 2 (Transcription):
# 1) Copy the MP4 videos into *your* S3 bucket.
# 2) Start an Amazon Transcribe job for each video.
# 3) Read each Transcribe JSON result back from S3 into a pandas DataFrame.

# Copy videos into your own S3 bucket
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/ s3://{bucket}/input/ --recursive

import boto3, json, uuid
from time import sleep
import pandas as pd

s3 = boto3.client("s3")
transcribe = boto3.client("transcribe")

transcripts = []

for obj in s3.list_objects_v2(Bucket=bucket, Prefix="input/")["Contents"]:
    key = obj["Key"]
    if key.endswith("/"):
        continue

    media_uri = f"s3://{bucket}/{key}"
    job_name = f"transcribe-{uuid.uuid4()}"
    output_key = f"transcribed/{key.split('/')[-1].replace('.mp4','.json')}"

    transcribe.start_transcription_job(
        TranscriptionJobName=job_name,
        Media={"MediaFileUri": media_uri},
        MediaFormat="mp4",
        LanguageCode="en-US",
        OutputBucketName=bucket,
        OutputKey=output_key
    )

    print("Started transcription:", job_name)
    sleep(2)

print("Waiting for jobs to complete...")
sleep(60)

for obj in s3.list_objects_v2(Bucket=bucket, Prefix="transcribed/")["Contents"]:
    resp = s3.get_object(Bucket=bucket, Key=obj["Key"])
    data = json.load(resp["Body"])
    transcripts.append({
        "Video": obj["Key"].split("/")[-1].replace(".json",".mp4"),
        "Transcription": data["results"]["transcripts"][0]["transcript"]
    })

df = pd.DataFrame(transcripts)
df.head()


## 1.6 Normalizing the text

In [ ]:
# AWS Step 3 (Normalization):
# Clean up transcript text so downstream NLP (key phrases/entities) is more consistent.

import re

def normalize_text(text):
    text = re.sub(r"http\S+", "", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["Transcription_normalized"] = df["Transcription"].apply(normalize_text)
df.head()


## 1.7 Extracting key phrases and topics

In [ ]:
# AWS Step 4 (Key phrases + topics):
# We run Amazon Comprehend batch jobs:
# - Key Phrases Detection (key phrases)
# - Entities Detection (acts as lightweight 'topics' like ORG/LOC/etc.)

import io, uuid

comprehend = boto3.client("comprehend")
s3_resource = boto3.resource("s3")

prefix = "capstone"
csv_buffer = io.StringIO()
df["Transcription_normalized"].str.slice(0,5000).to_csv(csv_buffer, header=False, index=False)

input_key = f"{prefix}/comprehend/input.csv"
s3_resource.Bucket(bucket).Object(input_key).put(Body=csv_buffer.getvalue())

input_s3_uri = f"s3://{bucket}/{input_key}"

job_id = str(uuid.uuid4())

comprehend.start_key_phrases_detection_job(
    InputDataConfig={"S3Uri": input_s3_uri, "InputFormat": "ONE_DOC_PER_LINE"},
    OutputDataConfig={"S3Uri": f"s3://{bucket}/"},
    DataAccessRoleArn=job_data_access_role,
    JobName=f"kpe-{job_id}",
    LanguageCode="en"
)

comprehend.start_entities_detection_job(
    InputDataConfig={"S3Uri": input_s3_uri, "InputFormat": "ONE_DOC_PER_LINE"},
    OutputDataConfig={"S3Uri": f"s3://{bucket}/"},
    DataAccessRoleArn=job_data_access_role,
    JobName=f"entities-{job_id}",
    LanguageCode="en"
)

print("Comprehend jobs started")


## 1.8 Creating the dashboard

In [ ]:
# AWS Step 5 (Dashboard):
# Provide a simple search function over the normalized transcripts.
# In a fuller solution, this could be a web UI or OpenSearch/Kibana dashboard.

# Simple in-notebook search dashboard

def search_videos(keyword):
    keyword = keyword.lower()
    return df[df["Transcription_normalized"].str.contains(keyword, na=False)][["Video","Transcription"]]

# Example search
search_videos("machine learning")


## 2. Congratulations!
You have completed the NLP Capstone Project.
Submit the lab to record your progress.
